# 5) Geospatial prep and choropleth map


In [ ]:
# Inputs: processed/merged | Process: load satnav (or build from latest) | Outputs: satnav_df
import pandas as pd
from pathlib import Path
cand = sorted(Path("../data/processed").glob("satnav_*.csv"))
if cand:
    satnav_df = pd.read_csv(cand[-1])
else:
    satcat_df = pd.read_csv(sorted(Path("../data/processed").glob("satcat_*.csv"))[-1])
    gp_df = pd.read_csv(sorted(Path("../data/processed").glob("gp_*.csv"))[-1])
    satnav_df = pd.merge(satcat_df, gp_df, on='NORAD_CAT_ID', how='left')
print(satnav_df.shape)


In [ ]:
# Inputs: world shapefile + satnav_df | Process: ISO mapping + counts + map | Outputs: folium map
import geopandas as gpd
import folium

world = gpd.read_file("../data/raw/ne_10m_admin_0_countries_ind.zip").to_crs("EPSG:4326")

# Map provider tags to ISO3 (~same as in main notebook)
rename_columns = {"US":"USA","UK":"GBR","PRC":"CHN","GER":"DEU","FGER":"DEU","CIS":"RUS"}
satnav_df["COUNTRY_ISO3"] = satnav_df.get("COUNTRY_ISO3", satnav_df.get("COUNTRY")).map(rename_columns).fillna(satnav_df.get("COUNTRY_ISO3", satnav_df.get("COUNTRY")))

counts = (satnav_df.dropna(subset=["COUNTRY_ISO3"]).groupby("COUNTRY_ISO3")["NORAD_CAT_ID"].nunique().rename("sat_count").reset_index())
world_std = world.rename(columns={"ISO_A3":"country_iso3","NAME":"country_name"})[["country_iso3","country_name","geometry"]]
world_summary = world_std.merge(counts, left_on="country_iso3", right_on="COUNTRY_ISO3", how="left").fillna({"sat_count":0})

m = folium.Map(location=[20,0], zoom_start=2, tiles="cartodbpositron")
folium.Choropleth(
    geo_data=world_summary.to_json(),
    data=world_summary,
    columns=["country_iso3","sat_count"],
    key_on="feature.properties.country_iso3",
    fill_color="YlOrRd",
    fill_opacity=0.5,
    line_opacity=0.3,
    legend_name="Satellites per country"
).add_to(m)
folium.GeoJson(world_summary.to_json(), name="labels", tooltip=folium.features.GeoJsonTooltip(fields=["country_name","sat_count"], aliases=["Country","Satellites"]))
folium.LayerControl().add_to(m)
m


In [11]:
# Create the GeoJSON with the launch sites and a small helper module for GeoPandas users.

import json
from pathlib import Path

# 1) Launch sites dictionary (EPSG:4326 WGS84)
launch_sites = {
    "TTMTR": {"site_name": "Tyuratam (Baikonur Cosmodrome), Kazakhstan", "lat": 45.9647, "lon": 63.3050},
    "AFETR": {"site_name": "Air Force Eastern Test Range (Cape Canaveral/KSC), Florida, USA", "lat": 28.4889, "lon": -80.5778},
    "AFWTR": {"site_name": "Air Force Western Test Range (Vandenberg SFB), California, USA", "lat": 34.7420, "lon": -120.5720},
    "WLPIS": {"site_name": "Wallops Island, Virginia, USA", "lat": 37.9400, "lon": -75.4660},
    "KYMTR": {"site_name": "Kapustin Yar Missile & Space Complex, Russia", "lat": 48.5720, "lon": 45.7350},
    "HGSTR": {"site_name": "Hammaguira Range, Algeria", "lat": 30.7800, "lon": -3.0600},
    "PKMTR": {"site_name": "Plesetsk Missile & Space Complex, Russia", "lat": 62.9256, "lon": 40.5778},
    "SNMLP": {"site_name": "San Marco Launch Platform (off Kenya), Indian Ocean", "lat": -2.9340, "lon": 40.2130},
    "WOMRA": {"site_name": "Woomera, South Australia, Australia", "lat": -31.1440, "lon": 136.8170},
    "KSCUT": {"site_name": "Uchinoura Space Center (Kagoshima), Japan", "lat": 31.2510, "lon": 131.0810},
    "FRGUI": {"site_name": "Europe’s Spaceport (CSG), Kourou, French Guiana", "lat": 5.2390, "lon": -52.7680},
    "JSC":   {"site_name": "Jiuquan Satellite Launch Center, China", "lat": 40.9600, "lon": 100.2980},
    "TNSTA": {"site_name": "Tanegashima Space Center, Japan", "lat": 30.3750, "lon": 130.9660},
    "SRI":   {"site_name": "Satish Dhawan Space Centre (Sriharikota), India", "lat": 13.7330, "lon": 80.2350},
    "XSC":   {"site_name": "Xichang Satellite Launch Center, China", "lat": 28.2460, "lon": 102.0270},
    "TSC":   {"site_name": "Taiyuan Satellite Launch Center, China", "lat": 37.5930, "lon": 112.9540},
    "YAVNE": {"site_name": "Palmachim (Yavne) Launch Facility, Israel", "lat": 31.8870, "lon": 34.6880},
    "SVOB":  {"site_name": "Svobodny (decommissioned) / Vostochny region predecessor, Russia", "lat": 51.4000, "lon": 128.1500},
    "CAS":   {"site_name": "Canaries Airspace (representative center)", "lat": 28.0000, "lon": -15.5000},
    "WRAS":  {"site_name": "Western Range Airspace (representative center)", "lat": 33.5000, "lon": -124.0000},
    "ERAS":  {"site_name": "Eastern Range Airspace (representative center)", "lat": 28.5000, "lon": -73.5000},
    "SADOL": {"site_name": "Unlisted/unknown in official SATCAT launch-sites (verify source)", "lat": None, "lon": None},
    "SEAL":  {"site_name": "Sea Launch (equatorial operations near 154°W)", "lat": 0.0000, "lon": -154.0000},
    "KWAJL": {"site_name": "U.S. Army Kwajalein Atoll, Marshall Islands", "lat": 9.3980, "lon": 167.4760},
    "KODAK": {"site_name": "Pacific Spaceport Complex–Alaska (Kodiak), USA", "lat": 57.4360, "lon": -152.3370},
    "OREN":  {"site_name": "Orenburg (Dombarovsky/Yasny) Launch Site, Russia", "lat": 51.0880, "lon": 59.8450},
    "SEM":   {"site_name": "Semnan Launch Site, Iran", "lat": 35.2340, "lon": 53.9200},
    "YUN":   {"site_name": "Sohae (Tongchang-ri) Satellite Launching Station, DPRK", "lat": 39.6600, "lon": 124.7050},
    "NSC":   {"site_name": "Naro Space Center (Goheung), Republic of Korea", "lat": 34.4300, "lon": 127.5350},
    "VOSTO": {"site_name": "Vostochny Cosmodrome, Russia", "lat": 51.8840, "lon": 128.3330},
    "WSC":   {"site_name": "Wenchang Space Launch Site (Hainan), China", "lat": 19.6140, "lon": 110.9510},
    "RLLC":  {"name": "Rocket Lab LC‑1, Māhia Peninsula, New Zealand", "lat": -39.2620, "lon": 177.8650},
    "YSLA":  {"name": "Yellow Sea Launch Area, China (representative center)", "lat": 34.9000, "lon": 121.2000},
    "SMTS":  {"name": "Shahrud Missile Test Site, Iran", "lat": 36.4000, "lon": 55.0200},
    "JJSLA": {"name": "Jeju Island Sea Launch Area, Republic of Korea (repr.)", "lat": 32.0000, "lon": 127.0000},
    "SCSLA": {"name": "South China Sea Launch Area, China (representative center)", "lat": 18.0000, "lon": 112.0000},
}


In [6]:

# 2) Build GeoJSON FeatureCollection
features = []
for code, info in launch_sites.items():
    lat, lon = info["lat"], info["lon"]
    # GeoJSON allows geometry to be null. We'll keep unlocated sites (e.g., SADOL) as null geometry.
    geometry = None if lat is None or lon is None else {"type": "Point", "coordinates": [lon, lat]}
    features.append({
        "type": "Feature",
        "properties": {"code": code, "name": info["name"]},
        "geometry": geometry
    })

geojson = {"type": "FeatureCollection", "name": "launch_sites", "crs": {"type": "name", "properties": {"name": "EPSG:4326"}}, "features": features}


In [8]:

# 3) Save GeoJSON
out_path = Path("../data/processed/launch_sites.geojson")
out_path.write_text(json.dumps(geojson, ensure_ascii=False, indent=2))


10119